<a href="https://colab.research.google.com/github/phamtuanlinh227-collab/python_for_chemistry/blob/master/Weekend_Projects_PHASE_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install deepchem rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 53.3 MB/s eta 0:00:00


In [3]:
import deepchem as dc
import torch
import torch.nn as nn
import warnings
from rdkit import RDLogger

# Bịt mõm hệ thống cảnh báo chung của Python
warnings.filterwarnings('ignore')

# Khâu mỏ thằng RDKit lại, cấm xả rác (Warning) ra Terminal
RDLogger.DisableLog('rdApp.*')

featurizer = dc.feat.CircularFingerprint(size=1024) # circularfingerprint like getmorganfingerpintasbitvect
loader = dc.data.CSVLoader(
    tasks=['proba'],
    feature_field = 'SMILES',
    featurizer=featurizer
) # two input X = 'SMILES', y = probability
dataset = loader.create_dataset('/content/drive/MyDrive/vscode/chem_master_code/proba_data.csv') # my file probability of 1000 smiles pass ro5

class ClassificationModel(torch.nn.Module):
  def __init__(self):
    super(ClassificationModel,self).__init__()
    self.dense1 = torch.nn.Linear(1024,1000)
    self.dense2 = torch.nn.Linear(1000,1)

  def forward(self, inputs):
    y = torch.nn.functional.relu(self.dense1(inputs))
    y = torch.nn.functional.dropout(y, p=0.5, training=self.training)
    logits = self.dense2(y)
    output = torch.sigmoid(logits)
    return output, logits

torch_model = ClassificationModel()
output_types = ['prediction', 'loss'] # output loss for machine learning, prediction for user
model = dc.models.TorchModel(torch_model, dc.models.losses.SigmoidCrossEntropy(), output_types=output_types)

tasks, datasets, transformers = dc.molnet.load_bace_classification(feturizer='ECFP', splitter='scaffold') # ECPF convert molecular as bit vect
train_dataset, valid_dataset, test_dataset = datasets
model.fit(train_dataset, nb_epoch=100)
metric = dc.metrics.Metric(dc.metrics.roc_auc_score)
print('training set score:', model.evaluate(train_dataset, [metric]))
print('test set score:', model.evaluate(test_dataset, [metric]))



training set score: {'roc_auc_score': np.float64(0.9996256198924356)}
test set score: {'roc_auc_score': np.float64(0.7573369565217392)}
